# 02 — XLE Systematic Momentum: pre-paper review

> Revisión exploratoria previa a paper trading. La señal, los factores y sus parámetros se mantienen congelados; este cuaderno no transmite órdenes.

## <b>Reproduction and friction sensitivity</b>

La notebook de ejecución se carga en memoria hasta su tabla de resumen. Esto conserva la identidad de señal ya auditada y permite variar solamente el supuesto ilustrativo de slippage.

In [ ]:
from pathlib import Path
import json

import numpy as np
import polars as pl

def candidate_root(start: Path) -> Path:
    for parent in (start.resolve(), *start.resolve().parents):
        direct = parent / 'Momentum_systematic_strategies' / '02_factor_momentum' / 'strategy_candidates' / 'xle_systematic_momentum'
        if (direct / 'notebooks' / '01_execution_feasibility.ipynb').exists():
            return direct
        candidate = parent / 'strategy_candidates' / 'xle_systematic_momentum'
        if (candidate / 'notebooks' / '01_execution_feasibility.ipynb').exists():
            return candidate
        if (parent / 'notebooks' / '01_execution_feasibility.ipynb').exists():
            return parent
    raise FileNotFoundError('No se encontró la candidata XLE.')

CANDIDATE_ROOT = candidate_root(Path.cwd())
source_path = CANDIDATE_ROOT / 'notebooks' / '01_execution_feasibility.ipynb'
source = json.loads(source_path.read_text(encoding='utf-8'))
state = {'__name__': '__pre_paper_review__'}
for cell in source['cells']:
    if cell['cell_type'] == 'code':
        code = ''.join(cell['source'])
        exec(compile(code, str(source_path), 'exec'), state)
        if 'execution_summary =' in code:
            break

schedule = state['execution_schedule']
run_execution = state['run_execution']
scenarios = state['SCENARIOS']
strategies = state['STRATEGIES']
initial_capital = state['INITIAL_CAPITAL']
xle_ohlc = state['xle_ohlc']
state['signal_reproduction_audit']

In [ ]:
def execution_metrics(history, trades):
    nav = history.get_column('nav').to_numpy()
    log_returns = np.diff(np.log(nav))
    drawdown = nav / np.maximum.accumulate(nav) - 1
    return {
        'final_capital': nav[-1],
        'cagr_pct': ((nav[-1] / initial_capital) ** (252 / len(log_returns)) - 1) * 100,
        'annualized_volatility_pct': log_returns.std(ddof=1) * np.sqrt(252) * 100,
        'sharpe': log_returns.mean() / log_returns.std(ddof=1) * np.sqrt(252),
        'max_drawdown_pct': drawdown.min() * 100,
        'operations': trades.height,
        'commission_usd': trades.get_column('commission').sum() if trades.height else 0.0,
        'slippage_usd': trades.get_column('slippage_cost').sum() if trades.height else 0.0,
    }

friction_rows = []
for bps in (0.0, 5.0, 10.0, 20.0):
    scenario = f'Review fixed + {bps:g} bps'
    scenarios[scenario] = {'commission': True, 'slippage_bps': bps}
    history, trades = run_execution(schedule, 'Systematic momentum', scenario)
    friction_rows.append({'slippage_bps': bps, **execution_metrics(history, trades)})

friction_review = pl.DataFrame(friction_rows).with_columns(pl.all().exclude('operations').round(3))
friction_review

## <b>Timing and accounting audit</b>

> La sensibilidad de un día conserva cada fecha de decisión y desplaza únicamente la apertura de ejecución a la siguiente sesión disponible.

In [ ]:
base_scenario = 'Review fixed + 5 bps'
delayed_schedule = (
    schedule.select('signal_date', 'tsmom_weight', 'market_weight', 'systematic_weight', 'signal_close',
                    pl.col('return_date').shift(-1).alias('return_date'))
    .drop_nulls()
    .join(xle_ohlc.rename({'session_date': 'return_date'}), on='return_date', how='inner')
    .sort('signal_date')
)

timing_rows = []
for strategy in strategies:
    scheduled_history, scheduled_trades = run_execution(schedule, strategy, base_scenario)
    late_history, late_trades = run_execution(delayed_schedule, strategy, base_scenario)
    scheduled = execution_metrics(scheduled_history, scheduled_trades)
    late = execution_metrics(late_history, late_trades)
    timing_rows.append({
        'strategy': strategy, 'scheduled_cagr_pct': scheduled['cagr_pct'],
        'one_session_late_cagr_pct': late['cagr_pct'],
        'cagr_change_pp': late['cagr_pct'] - scheduled['cagr_pct'],
        'scheduled_final_capital': scheduled['final_capital'],
        'late_final_capital': late['final_capital'],
    })
timing_review = pl.DataFrame(timing_rows).with_columns(pl.all().exclude('strategy').round(3))

accounting_audit = pl.DataFrame([{
    'schedule_rows': schedule.height,
    'signal_date_not_before_return_date': int((schedule['signal_date'] >= schedule['return_date']).sum()),
    'null_executable_prices': sum(schedule.select('open', 'close', 'signal_close').null_count().row(0)),
    'nonpositive_prices': int(((schedule['open'] <= 0) | (schedule['close'] <= 0) | (schedule['signal_close'] <= 0)).sum()),
    'dividend_events': xle_ohlc.filter(pl.col('dividend_amount').fill_null(0) != 0).height,
    'split_events': xle_ohlc.filter(pl.col('split_coefficient').fill_null(1) != 1).height,
}])
timing_review, accounting_audit

## <b>Descriptive uncertainty</b>

Un block bootstrap circular de 21 sesiones conserva dependencia local aproximada. Estima la incertidumbre de los diferenciales históricos observados, pero no corrige que XLE fue seleccionada después de examinar el mismo historial.

In [ ]:
returns = (
    state['reference_namespace']['strategy_returns']
    .filter(pl.col('asset') == 'XLE')
    .select('return_date', 'alternative', 'strategy_return')
    .pivot(on='alternative', index='return_date', values='strategy_return')
    .drop_nulls()
    .sort('return_date')
)
rng = np.random.default_rng(126)

def block_bootstrap(values, block_length=21, repetitions=2_000):
    samples, n = [], len(values)
    for _ in range(repetitions):
        index = []
        while len(index) < n:
            start = rng.integers(0, n)
            index.extend((start + np.arange(block_length)) % n)
        samples.append(values[np.asarray(index[:n])].mean() * 252 * 100)
    return np.asarray(samples)

bootstrap_rows = []
for benchmark in ('TSMOM total', 'Market momentum', 'Buy & Hold'):
    difference = (returns['Systematic momentum'] - returns[benchmark]).to_numpy()
    sampled = block_bootstrap(difference)
    bootstrap_rows.append({
        'benchmark': benchmark,
        'annualized_log_difference_pct': difference.mean() * 252 * 100,
        'bootstrap_p05_pct': np.quantile(sampled, 0.05),
        'bootstrap_median_pct': np.quantile(sampled, 0.50),
        'bootstrap_p95_pct': np.quantile(sampled, 0.95),
        'probability_positive': (sampled > 0).mean(),
    })
bootstrap_review = pl.DataFrame(bootstrap_rows).with_columns(pl.all().exclude('benchmark').round(3))
bootstrap_review

## <b>Operational findings before paper observation</b>

- La identidad de los pesos debe continuar siendo exacta frente a la notebook congelada en cada ejecución.
- La simulación histórica dimensiona inicialmente con el cierre de señal, pero reduce una compra con la apertura observada si no resulta financiable. Ese ajuste conoce el precio de fill y no puede convertirse directamente en una orden preabierta.
- Los precios de apertura históricos no garantizan un fill paper o live. La modalidad Market-on-Open debe observarse por separado y compararse contra el precio registrado.
- Dividendos y el split se contabilizan con las fechas y coeficientes del proveedor; faltan fecha de pago, impuestos y confirmaciones del broker.
- Gold no registra una marca de publicación/cierre definitivo del proveedor. Antes de toda observación hay que exigir un corte diario documentado, datos completos y validación del contrato/cash/órdenes abiertas.